# 공동연구 사전 실습 — 보고동사로 다시 읽기

버셀 담당자를 만나기 전에, 화학(황 교수님)과 언어학(윤 교수님)의 관점에서 이번 MVP를 직접 실행해 보고 검토하기 위한 노트북입니다. 코랩을 처음 쓰셔도 따라오실 수 있도록 단계마다 안내를 붙였습니다.

---
## 1. 코랩 접속
1. 브라우저에서 **[colab.research.google.com](https://colab.research.google.com)** 에 접속합니다.
2. 구글 계정으로 로그인합니다.

## 2. 노트북 올리기
1. 시작 화면의 **파일 업로드** 탭을 누릅니다.
2. **`보고동사로다시읽기_연구자사전실습.ipynb`** 를 고릅니다.

## 3. 셀 실행
- 셀을 누른 뒤 **`Shift + Enter`**, 또는 셀 왼쪽의 **▶ 버튼**을 누릅니다.
- **위에서부터 차례로** 실행합니다. 중간을 건너뛰면 뒤 셀에서 오류가 납니다.

## 4. 설치
- 「0. 설치」 셀을 가장 먼저 실행합니다(1~2분). `준비 완료` 메시지가 나오면 됩니다.

## 5. 자료 올리기
- 「1. 자료 올리기」 셀을 실행하면 **파일 선택** 버튼이 나옵니다. **`rv_colab.zip`** 을 고르면 자동으로 풀립니다.

## 6. 결과가 보이는 방식
| 보는 것 | 방법 | 모양 |
|---|---|---|
| 숫자, 표, 목록 | `print()` | 글자 |
| 색칠한 문장, 읽기 화면 | `show_sentence()`, `display()` | 형광펜 색칠 |

**주의**: 코랩 실행 환경의 파일은 영구 보관되지 않습니다. 실습이 끝나면 마지막 셀에서 결과와 검토 메모를 내려받아 주세요.

# 이번 사전 실습의 범위

**목적**: 화학 논문 두 편에서 **누가 무엇을 보았다고 말하는지**를 동사(보고동사, reporting verb)로 읽어 내는 MVP를 실행해 보고, 그 판정이 화학 문맥과 언어학 기준에서 타당한지 함께 검토합니다.

**대상 논문**(두 편 모두 CC BY 4.0 공개 논문)
| | 논문 | 주제 |
|---|---|---|
| A | Paiva Santiago et al. (2025), *ACS Omega* | 맥주 찌꺼기에서 초음파와 심층공융용매(DES)로 페놀 추출 |
| B | Bleus et al. (2024), *Molecules* | 맥주 찌꺼기·몰트 먼지에서 고온 수열 추출과 천연 심층공융용매(NADES) |

**사용 도구**
- `spaCy`: 품사와 문장 구조(의존 구문)를 분석하는 파이썬 라이브러리
- `rvtool`: 이번 MVP의 공통 처리 도구(입력 검사·저장·검증·읽기 화면)

이 노트북은 **외부 AI를 쓰지 않습니다.** 분석은 코랩 안에서 구문분석과 규칙으로 이루어집니다. AI API를 쓰는 판(② 노트북)은 별도이며 이번 실습 범위가 아닙니다. SLA(감성 분석)와도 관련이 없습니다.

---
# 판정 기준: 같은 동사, 다른 목소리

먼저 세 가지 대표적인 쓰임을 비교합니다. 실제 도구에서는 방법·해석·검토 필요 등을 포함해 여섯 역할로 구분합니다(아래 표).

```text
Chen et al. [32] reported the surface tension of 50 DESs.   ← 선행연구가 말한 것
The results showed a decrease in viscosity.                 ← 이 논문의 결과
Figure 2 shows the effect of temperature.                   ← 그림을 가리킴
```

Hyland(2001)는 보고(reporting)를 **명제 내용을 다른 출처에 귀속시키는 것**으로 정의했습니다. 엄밀한 의미의 보고동사는 첫 번째뿐입니다. 이번 MVP는 세 목소리를 모두 색칠해 구별합니다.

| 색 | 역할(도구의 분류) | 예 |
|---|---|---|
| 노랑 | 선행연구 보고 | *Chen et al. [32] **reported*** |
| 초록 | 자기 결과 제시 | *The results **showed*** |
| 보라 | 해석·주장 | *This **suggests** that* |
| 청회색 | 연구 수행·방법 | *Samples **were prepared*** |
| 회색 | 그림·표·본문 지시 | *Figure 2 **shows*** |
| 빗금 | 역할 검토 필요 | 규칙으로 결정하지 못한 경우 |

이 역할 구분은 이번 도구의 분류입니다. Hyland의 과정 분류인 연구행위·인지행위·담화행위, 그리고 각 과정 안에서의 평가적 선택(사실적·비사실적·잠정적 등)을 그대로 구현한 것은 아닙니다. 색은 **규칙이 붙인 후보**이고 사람이 확인한 판정이 아닙니다. 이번 실습의 핵심은 원문을 함께 읽고 이 판정의 타당성을 검토하는 것입니다.

---
# 0. 설치
**실행 목적**: 영어 문장의 품사와 구조를 분석할 도구를 준비합니다. 세션마다 한 번 실행합니다.

| 설치하는 것 | 무엇인가 |
|---|---|
| spaCy | 문장을 처리하고 언어 분석을 하는 파이썬 라이브러리 |
| en_core_web_sm | spaCy가 쓰는, 미리 학습된 소형 영어 모델(품사, 의존 구문, 기본형 찾기) |

"API 없음"은 외부 생성형 AI에 문장을 보내지 않는다는 뜻이지, 학습된 모델을 전혀 쓰지 않는다는 뜻은 아닙니다. 이 영어 모델은 코랩 안에서 실행됩니다. 일반 영어 문어 자료로 학습된 모델이라 화학 논문의 복잡한 문장에서는 분석이 틀릴 수 있습니다. 모델이 '선행연구 보고' 같은 역할을 직접 판정하는 것은 아니고, 모델의 구문 분석 결과를 `engine_local.py`의 규칙이 활용해 역할을 정합니다.
**성공 기준**: `준비 완료` 메시지와 spaCy 판 번호가 출력됩니다.

**검토할 점**
- 설치에서 오류가 나면 메시지 전체를 기록해 두세요. 버셀 세션에서 환경 설정을 물어볼 때 필요합니다.

In [1]:
print("설치 칸 생략(시험 환경)")

설치 칸 생략(시험 환경)


In [2]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("준비 완료 — spaCy", spacy.__version__)

준비 완료 — spaCy 3.8.18


---
# 1. 자료 올리기
**실행 목적**: 이번 실습에 필요한 9개 파일을 제자리에 둡니다.

| 묶음 | 파일 | 역할 |
|---|---|---|
| 논문 자료 4개 | `inputs/paper_a_input.txt`, `inputs/paper_b_input.txt` | 문장 ID와 본문("어떤 문장인가") |
| | `inputs/paper_a_structure.tsv`, `inputs/paper_b_structure.tsv` | 절·문단·순서·단어 수("논문의 어디에 있는가") |
| 처리 코드 2개 | `engine_local.py` | 표현 후보와 역할을 찾는 분석 엔진 |
| | `rvtool.py` | 입력 검사, 결과 검증·저장·다시 읽기, 집계, 화면 연결 |
| 검사 규격 2개 | `schemas/core_v1.3.json`, `schemas/whole_paper_hits_v0.1.json` | 결과가 정해진 형식을 지켰는지 확인하는 규칙(정답표 아님) |
| 화면 틀 1개 | `viewer_template.html` | 결과를 넣어 색칠·그래프·원문 연결 화면을 만드는 틀 |

TXT는 논문의 '본문', TSV는 그 본문의 '주소표'입니다. 같은 문장 ID로 둘을 연결하면 문장 내용과 위치를 함께 알 수 있어, 절별 그래프, 같은 문단 보기, 1,000단어당 비교가 가능해집니다. 이 단계의 메시지는 "준비물 확인"이며, 파일 내용이나 분석의 성공까지 확인한 것은 아닙니다. API 판(`engine_api.py`)과 초안 주석 CSV는 이번 실습에 필요하지 않습니다.
**성공 기준**: `필수 파일: 모두 있음`이 출력됩니다. 파일이 빠졌으면 이 셀에서 오류로 멈추니, ZIP을 다시 올리고 이 셀을 다시 실행하세요.

**검토할 점**
- 입력은 미리 전처리된 파일입니다. PDF 추출이나 DOI 수집은 이 MVP의 범위 밖입니다.

In [3]:
import os, zipfile, shutil, glob, tempfile, hashlib
NEED = ["rvtool.py", "engine_local.py", "viewer_template.html", "schemas/core_v1.3.json", "schemas/whole_paper_hits_v0.1.json",
        "inputs/paper_a_input.txt", "inputs/paper_a_structure.tsv", "inputs/paper_b_input.txt", "inputs/paper_b_structure.tsv"]
ZIP_NAME = "rv_colab.zip"   # 올릴 자료 ZIP 이름
def install_zip(zpath):
    """선택한 ZIP 하나만 새 임시 폴더에 풀고, 필요한 파일을 현재 폴더에 덮어써 옮긴다(이전 실습 파일과 섞이지 않게)."""
    tmp = tempfile.mkdtemp(prefix="rv_zip_")
    with zipfile.ZipFile(zpath) as f: f.extractall(tmp)
    base = os.path.join(tmp, "rv_colab") if os.path.isdir(os.path.join(tmp, "rv_colab")) else tmp
    lacking = [rel for rel in NEED if not os.path.exists(os.path.join(base, rel))]
    if lacking:   # 새 ZIP 자체에 필수 파일이 다 있어야 복사한다(이전 실습 파일이 빈자리를 채우지 않게)
        raise FileNotFoundError(f"올린 ZIP에 필수 파일이 없습니다: {lacking}. 아무것도 복사하지 않았습니다. 최신 rv_colab.zip을 올려 주세요.")
    for rel in NEED + ["annotations/paper_a_draft_v1.3.csv", "annotations/paper_b_draft_v1.3.csv"]:
        src = os.path.join(base, rel)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(rel) or ".", exist_ok=True); shutil.copy2(src, rel)
    print(f"자료 ZIP: {zpath} (SHA-256 {hashlib.sha256(open(zpath, 'rb').read()).hexdigest()[:16]}…)")
if os.path.exists(ZIP_NAME):
    install_zip(ZIP_NAME)
else:
    try:
        from google.colab import files
        print(f"{ZIP_NAME} 을 올려 주세요")
        up = [n for n in files.upload() if n.endswith(".zip")]
        if len(up) != 1: raise FileNotFoundError(f"ZIP 파일을 하나만 올려 주세요(올린 ZIP: {up})")
        install_zip(up[0])
    except ImportError:
        pass
missing = [p for p in NEED if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(f"필수 파일이 없습니다: {missing}. {ZIP_NAME}을 올린 뒤 이 셀을 다시 실행하세요.")
print("필수 파일: 모두 있음")

필수 파일: 모두 있음


---
# 2. 논문 불러오기
**전제**: 이번 실습은 전처리와 원문 대조를 마쳐 확정한 입력 판본을 사용합니다. 전체 구조에서는 앞단의 '전처리·검증'(본문 추출, 문장 분리, ID 부여, 원문 대조)이 확정 입력을 만들고, 지금의 분석 단계가 그것을 받아 분석·저장·탐색합니다.

**실행 목적**: 확정된 문장·구조 자료를 불러와 입력 계약(형식·ID·순서)을 검사하고, 입력 파일 머리부의 전처리 기록과 일치하는지 확인합니다. 기존 문장 경계와 ID는 유지하며 문장을 다시 나누거나 합치지 않습니다(단어 나누기와 구문 분석은 이후 단계에서 합니다).
**성공 기준**: 입력 검사를 통과하고, 전처리 기록과 실제로 읽은 값이 모두 "일치"로 나옵니다. 이번 판본의 확인값은 논문 A 249문장·64문단, 논문 B 215문장·48문단, 인용 표지 문장 각 49개입니다. 이 숫자는 이 입력 판본의 확인값이며 다른 논문에 쓰는 고정값이 아닙니다. 인용 표지 문장 49개가 보고동사 문장 49개라는 뜻도 아닙니다.

**검토할 점**
- 두 논문의 절 순서가 다릅니다(A: 서론→방법→결과·논의, B: 서론→결과·논의→방법). B처럼 방법 절을 뒤에 두는 것은 MDPI(Molecules) 저자 지침이 안내하는 구성입니다. 그래서 표현 분포를 비교할 때는 원문 안의 위치(그래프)와 절의 기능(절 유형별 표)을 나누어 봅니다.
- 문장 ID `3.2.3-007`은 3.2.3절의 7번째 문장이라는 뜻입니다.

In [4]:
import sys, importlib, html, re
sys.path.insert(0, ".")
import rvtool as rv, engine_local as L
importlib.reload(rv); importlib.reload(L)   # 다시 실행할 때 새로 받은 파일을 쓰도록
if not hasattr(rv, "save_validated_run"):
    raise RuntimeError(f"공통 모듈 판본이 이 노트북보다 오래되었습니다({rv.TOOL_VERSION}). 이 노트북과 같은 때 받은 rv_colab.zip을 올려 주세요.")
print("공통 모듈:", rv.TOOL_VERSION, "| 규칙:", L.RULE_VERSION)
A = rv.load_document("inputs/paper_a_input.txt", "inputs/paper_a_structure.tsv", "paper_a", document_version="preproc-v1.2-xml-20261015")
B = rv.load_document("inputs/paper_b_input.txt", "inputs/paper_b_structure.tsv", "paper_b", document_version="preproc-v1.2-xml-20261015")
KO = {"ABS": "초록", "INT": "서론", "MET": "방법", "RD": "결과·논의", "CON": "결론"}
for D, name in ((A, "논문 A"), (B, "논문 B")):
    d = rv.describe(D)
    print(f"{name}: 문장 {d['sentences']}개, 문단 {d['paragraphs']}개, 인용이 있는 문장 {d['citing_sentences']}개")
    print("   절별 문장 수 / sentences per section:", {KO[k]: v for k, v in d["by_section"].items()})
print("\n전처리 기록과 비교(입력 파일 머리부의 기록 ↔ 실제로 읽은 값)")
for D, name in ((A, "논문 A"), (B, "논문 B")):
    rows = rv.check_preprocessing_record(D)
    print("  " + name + ": " + ", ".join(f"{r['item']} {r['record']}↔{r['actual']} {r['status']}" for r in rows))
    if any(r["status"] == "불일치" for r in rows):
        raise ValueError(f"{name}: 전처리 기록과 읽은 값이 다릅니다. 같은 판본의 파일을 올렸는지 확인하세요.")

공통 모듈: rvtool-1.3.1 | 규칙: whole-paper-0.4.3
논문 A: 문장 249개, 문단 64개, 인용이 있는 문장 49개
   절별 문장 수 / sentences per section: {'초록': 10, '서론': 24, '방법': 67, '결과·논의': 139, '결론': 9}
논문 B: 문장 215개, 문단 48개, 인용이 있는 문장 49개
   절별 문장 수 / sentences per section: {'초록': 8, '서론': 43, '결과·논의': 73, '방법': 85, '결론': 6}

전처리 기록과 비교(입력 파일 머리부의 기록 ↔ 실제로 읽은 값)
  논문 A: 총 문장 수 249↔249 일치, 총 문단 수 64↔64 일치, 인용 표지 문장 수 49↔49 일치
  논문 B: 총 문장 수 215↔215 일치, 총 문단 수 48↔48 일치, 인용 표지 문장 수 49↔49 일치


---
# 색칠 도구 준비
**실행 목적**: 문장을 형광펜으로 칠해 보여 주는 `show_sentence()`를 만듭니다. 먼저 실행하고, 필요하면 코드에서 색과 역할 이름의 대응을 확인하세요.
**성공 기준**: `show_sentence() 준비 완료`가 출력됩니다.

**검토할 점**
- 표에 나오는 '주어'는 구문분석으로 찾은 행위자 후보입니다.

In [5]:
from IPython.display import HTML, display
COLOR = {"prior": "#F1DC5A", "result": "#BFE3C4", "interpret": "#DDD0F2", "method": "#CFDDEA", "ref": "#DADDD9", "uncertain": "#F6E1BE"}
ROLE_KO = {"prior": "선행연구 보고", "result": "자기 결과 제시", "interpret": "해석·주장", "method": "연구 수행·방법", "ref": "그림·표·본문 지시", "uncertain": "역할 검토 필요"}
def analyze_text(text, sid="9-001"):
    """논문에 없는 문장도 분석할 수 있게 해 줍니다 / analyze any sentence you type"""
    s = rv.Sentence(sid, 1, text, "RD", "9", "practice", "P001", 1, len(text.split()))
    core, hits, excluded = L.analyze_sentence("practice", s, "en_core_web_sm")
    return core, hits
def show_sentence(text, hits=None, title=""):
    if hits is None: _, hits = analyze_text(text)
    out, cur = "", 0
    for h in sorted(hits, key=lambda h: h["start"]):
        out += html.escape(text[cur:h["start"]])
        out += f'<mark style="background:{COLOR[h["role"]]};padding:0 2px">{html.escape(h["text"])}</mark>'
        cur = h["end"]
    out += html.escape(text[cur:])
    out = re.sub(r"\[(\d[\d,\s–−-]*)\]", r'<span style="color:#2E5AA8;font-family:monospace">[\1]</span>', out)
    rows = "".join(f'<tr><td><b>{html.escape(h["text"])}</b></td><td style="background:{COLOR[h["role"]]}">{ROLE_KO[h["role"]]}</td><td>{html.escape(h.get("agent",""))}</td></tr>' for h in hits)
    display(HTML(f'<div style="font-family:Georgia,serif;font-size:17px;line-height:1.7;margin:6px 0">{("<small>"+html.escape(title)+"</small><br>") if title else ""}{out}</div>'
                 + (f'<table style="font-size:13px"><tr><th>동사 / verb</th><th>역할 / role</th><th>주어 / who</th></tr>{rows}</table>' if hits else "<small>찾은 동사가 없습니다 / no verbs found</small>")))
print("show_sentence() 준비 완료")

show_sentence() 준비 완료


---
# 실습 1. 한 문장 읽기
**예문**(논문 A, 3.2.3-007): *… Chen et al.[32] reported the surface tension of 50 DESs …*
- `Chen et al.` 다른 연구자, `[32]` 참고문헌 32번, `reported` 그들이 보고함 → 노랑(선행연구 보고)

**실행 목적**: 규칙이 한 문장을 어떻게 판정하는지 봅니다.
**성공 기준**: `reported`가 노랑(선행연구 보고)으로 칠해지고, 주어 칸에 '선행연구'가 표시됩니다. 셀 아래에 기대값과 실제 출력의 비교가 함께 나옵니다. 주어 칸은 구문분석으로 찾은 후보라서 문장에 따라 '불명'이 나올 수 있습니다. 그럴 때는 검토 메모에 남겨 주세요.

**검토할 점**
- 이 판정은 화학 문맥에서 타당한가?
- 직접 문장을 바꿔 넣어 보고(아래 `my_sentence`), 판정이 흔들리는 경우를 찾아보세요.

In [6]:
# 논문 A의 3.2.3-007 문장을 꺼내서 색칠해 봅니다 / take sentence 3.2.3-007 from paper A
sentence = A.by_id()["3.2.3-007"].text
show_sentence(sentence, title="논문 A · 3.2.3-007")
_, hits = analyze_text(sentence, "3.2.3-007")
ok = any(h["text"] == "reported" and h["role"] == "prior" and h["agent"] == "선행연구" for h in hits)
print("기대: reported = 선행연구 보고, 주어 = 선행연구 →", "일치" if ok else f"불일치 {[(h['text'], ROLE_KO[h['role']], h['agent']) for h in hits]}")

동사 / verb,역할 / role,주어 / who
reported,선행연구 보고,선행연구


기대: reported = 선행연구 보고, 주어 = 선행연구 → 일치


In [7]:
# ── 직접 확인 ─────────────────────────────
# 따옴표 안의 문장을 바꿔 판정을 확인해 보세요
my_sentence = "Smith et al. [5] demonstrated that the yield increased at higher temperatures."

show_sentence(my_sentence)

동사 / verb,역할 / role,주어 / who
demonstrated,선행연구 보고,선행연구


---
# 실습 2. 같은 *show*, 다른 역할
| 문장 | 누가 | 역할 |
|---|---|---|
| *Wang et al. [55] **showed** …* | 다른 연구자 | 선행연구 보고 |
| *The results **showed** …* | 이 논문의 자료 | 자기 결과 제시 |
| *Figure 2 **shows** …* | 그림 | 그림·표 지시 |

동사만으로는 역할을 정할 수 없습니다. 규칙은 **주어**와 **인용 번호**를 함께 봅니다.

**실행 목적**: 이 도구가 존재하는 이유인 '같은 동사, 다른 역할'을 확인합니다.
**성공 기준**: 세 문장이 각각 노랑·초록·회색으로 칠해집니다.

**검토할 점**
- 선행연구 보고와 현재 연구 서술을 잘 구분했는가?
- 화학 논문에서 이 셋 말고 다른 목소리(예: 일반 사실, 교과서적 지식)는 어떻게 다뤄야 하는가?

In [8]:
examples = [
    "Wang et al. [55] showed that the solvent could be recovered.",
    "The results showed a substantial decrease in viscosity.",
    "Figure 2 shows the effect of temperature on the density.",
]
for e in examples:
    show_sentence(e)

동사 / verb,역할 / role,주어 / who
showed,선행연구 보고,선행연구
could be recovered,연구 수행·방법,연구 대상(물질·시료)


동사 / verb,역할 / role,주어 / who
showed,자기 결과 제시,자료·측정


동사 / verb,역할 / role,주어 / who
shows,그림·표·본문 지시,그림·표


In [9]:
# ── 직접 확인 ─────────────────────────────
# 같은 동사(found)로 세 역할을 만들어 판정을 확인합니다
my_prior  = "Lee et al. [12] found that water lowers the viscosity."
my_result = "We found that the extraction yield was highest at 60 °C."
my_figure = "The details are found in Table 3."

for s in (my_prior, my_result, my_figure):
    show_sentence(s)

동사 / verb,역할 / role,주어 / who
found,선행연구 보고,선행연구


동사 / verb,역할 / role,주어 / who
found,자기 결과 제시,필자


동사 / verb,역할 / role,주어 / who
are found,그림·표·본문 지시,기타


---
# 실습 3. 인용 번호 읽기
| 표기 | 뜻 |
|---|---|
| `[7]` | 7 |
| `[40],[41]` | 40, 41 |
| `[23–25]` | 23, 24, 25 |
| `[23]−[25]` | 23, 24, 25 |

**실행 목적**: 인용 번호를 코드로 정규화하는 방식을 확인합니다. 원문은 바꾸지 않고 분석용 값만 만듭니다.
**성공 기준**: 위 표와 같은 결과가 출력됩니다.

**검토할 점**
- 두 저널(ACS·MDPI)의 인용 표기 방식 차이가 결과에 영향을 주는가?

In [10]:
for t in ["one source [7].", "two sources [40],[41].", "a range [23–25].", "another range [23]−[25]."]:
    print(f"{t:28s} → {rv.citation_numbers(t)}")

one source [7].              → [7]
two sources [40],[41].       → [40, 41]
a range [23–25].             → [23, 24, 25]
another range [23]−[25].     → [23, 24, 25]


In [11]:
# ── 직접 확인 ─────────────────────────────
my_citation = "as shown in several studies [3,8–10]."
print(rv.citation_numbers(my_citation))

[3, 8, 9, 10]


---
# 실습 4. 논문 전체 분석
**실행 목적**: 두 논문의 모든 문장을 분석하고, 결과의 형식·문장 ID·표현 위치를 검증한 뒤 문제가 없을 때만 저장합니다(몇 초). 실행마다 새 폴더에 저장되고 덮어쓰지 않으며, 규칙으로 제외한 후보와 그 사유도 함께 보존합니다.
**성공 기준**: 두 논문 모두 `검증 통과·저장`과 역할별 개수가 출력됩니다(참고: A 188개, B 182개).

**검토할 점**
- 역할별 분포가 화학 연구자의 직관과 맞는가? 특히 '연구 수행·방법'이 많은 이유를 원문으로 확인해 보세요.
- 아래 `my_id`에 문장 번호를 넣어 개별 문장을 확인할 수 있습니다.

In [12]:
import collections
RUN_ROOT = "rv_runs"
RUNS = {}
for D, name in ((A, "논문 A"), (B, "논문 B")):
    core, whole, excluded, info = L.analyze(D)
    rid = rv.new_run_id("research-practice")
    rec = rv.base_record(D, "local", rid, ["core_v1.3", "whole_paper_hits_0.1"], **info)
    # 저장 전에 core·whole을 검증하고, 문제가 있으면 저장하지 않고 멈춥니다. 제외 후보는 excluded.json으로 함께 보존합니다.
    rv.save_validated_run(RUN_ROOT, D, rid, rec, core, whole, extra=dict(excluded=excluded))
    RUNS[D.document_id] = rid
    c = collections.Counter(h["role"] for h in whole)
    print(f"{name}: 검증 통과·저장 ({rid}) | 색칠한 동사 {len(whole)}개, 규칙으로 제외한 후보 {len(excluded)}개")
    for r in ["prior", "result", "interpret", "method", "ref", "uncertain"]:
        print(f"   {ROLE_KO[r]:10s} {c[r]:4d}")

논문 A: 검증 통과·저장 (20261102-202746-research-practice-ae2523) | 색칠한 동사 188개, 규칙으로 제외한 후보 87개
   선행연구 보고      29
   자기 결과 제시     34
   해석·주장        14
   연구 수행·방법     96
   그림·표·본문 지시    8
   역할 검토 필요      7


논문 B: 검증 통과·저장 (20261102-202747-research-practice-3ed28b) | 색칠한 동사 182개, 규칙으로 제외한 후보 105개
   선행연구 보고      21
   자기 결과 제시     18
   해석·주장        12
   연구 수행·방법    115
   그림·표·본문 지시    4
   역할 검토 필요     12


In [13]:
# ── 직접 확인 ─────────────────────────────
# 논문 A에서 확인할 문장 번호
my_id = "3.3.2-020"

r = rv.load_run(RUN_ROOT, "paper_a", RUNS["paper_a"])
hits = [h for h in r["whole"] if h["sentence_id"] == my_id]
show_sentence(A.by_id()[my_id].text, hits, title=f"논문 A · {my_id}")

동사 / verb,역할 / role,주어 / who
indicating,해석·주장,자료·측정


---
# 실습 5. 용례 찾기(콘코던스, KWIC)
찾는 말을 가운데 두고 앞뒤 문맥을 나란히 보여 줍니다.

**실행 목적**: 특정 동사가 논문 전체에서 어떤 문맥에 쓰이는지 모아 봅니다.
**성공 기준**: 용례 목록과 "전체 n개 중 m개 표시"가 출력됩니다. 부분 문자열 검색이라 `show`는 shows·showed도 찾습니다.

**검토할 점**
- 같은 동사의 용례를 나란히 놓았을 때 규칙의 판정 기준이 일관적인가?
- 불확실한 사례에서 어떤 문맥(앞 문장, 문단)이 더 필요한가?

In [14]:
def kwic(D, word, width=55, limit=30):
    """부분 문자열 검색입니다(단어 단위 아님). 예: 'show'는 shows·showed 안의 show도 찾습니다."""
    found, shown = 0, 0
    for s in D.sentences:
        low = s.text.lower(); p = low.find(word.lower())
        while p >= 0:
            found += 1
            if shown < limit:
                left, mid, right = s.text[max(0, p-width):p], s.text[p:p+len(word)], s.text[p+len(word):p+len(word)+width]
                print(f"{s.id:>10}  {left:>{width}} | {mid} | {right}"); shown += 1
            p = low.find(word.lower(), p + 1)
    print(f"\n'{word}': 전체 {found}개 중 {shown}개 표시(부분 문자열 검색) / {found} found, {shown} shown (substring search)")
kwic(A, "reported")

     1-012  aric acid (p- CA) are the types of HCAs that have been  | reported |  most frequently in the extraction of phenolic compound
   2.3-012   Aqueous solutions were evaluated because some studies  | reported |  that water reduces the viscosity of DES, contributing 
 2.4.2-002                                The water content values  | reported |  are the averages of at least three measurements.
 3.2.3-007  erature on the surface tension of DES, Chen et al.[32]  | reported |  the surface tension of 50 DESs, two of which have comp
 3.2.3-012                         For example, Aravena et al.[33]  | reported |  values of γ = 64.9 mN/m at 40 °C, and Chanioti and Tzi
 3.3.1-012                                      Other studies also  | reported |  that ChCl:Gly added with water was more effective in e

'reported': 전체 6개 중 6개 표시(부분 문자열 검색) / 6 found, 6 shown (substring search)


In [15]:
# ── 직접 확인 ─────────────────────────────
# 찾을 말과 논문(A 또는 B)
my_word  = "showed"
my_paper = B          # A 또는 B / A or B

kwic(my_paper, my_word)

   2.2-003      Rheology measurements (dynamic viscosity) at 25 °C  | showed |  a substantially higher viscosity of water-free MA comp
   2.3-006                            With a recovery of >97%, SYR  | showed |  the highest degradation in function of time.
   2.4-005   single deviating result of MD-MAC-GLY, the HTE method  | showed |  improved overall extraction yields ranging from 56% (M

'showed': 전체 3개 중 3개 표시(부분 문자열 검색) / 3 found, 3 shown (substring search)


---
# 실습 6. 읽기 화면
색칠 → 흐름 그래프 → 클릭 → 원래 문맥으로 읽는 화면입니다.

| 화면에서 | 하는 일 |
|---|---|
| 꺾은선 그래프의 점 클릭 | 그 구간의 문장이 열림 |
| 문장 클릭 | 앞뒤 문장과 문단, 판정 근거(역할·귀속·주어) |
| 용례 찾기 | 콘코던스 |
| 두 논문 나란히 | 문서 내 위치와 절 유형별 비교 |

화면이 코랩 안에서 좁게 보이면 내려받은 HTML 파일을 브라우저로 여세요.

**실행 목적**: 버셀에서 서비스로 만들려는 화면의 시안을 실제 결과로 확인합니다.
**성공 기준**: 화면이 표시되고 HTML 파일이 내려받아집니다.

**검토할 점**
- 이 결과가 실제 논문 읽기에 도움이 되는가?
- 버셀 담당자에게 어떤 구현 도움을 요청해야 하는가?(예: 파일 업로드, 결과 저장, 여러 사용자의 검토 기록)

In [16]:
import time
view = f"보고동사로다시읽기_연구자화면_{time.strftime('%H%M%S')}.html"
rv.build_viewer([rv.load_run(RUN_ROOT, "paper_a", RUNS["paper_a"]), rv.load_run(RUN_ROOT, "paper_b", RUNS["paper_b"])], [A, B], view)
print("화면 파일 / screen file:", view)
try:
    from google.colab import files
    display(HTML(open(view, encoding="utf-8").read()))
    files.download(view)
except ImportError:
    pass

화면 파일 / screen file: 보고동사로다시읽기_연구자화면_202747.html


---
# 실습 7. 두 논문 비교
**실행 목적**: 두 논문을 1,000단어당 수치로 비교합니다(초록 제외).
**성공 기준**: 역할 × 절 유형 표가 출력됩니다.

**검토할 점**
- 선행연구 보고는 어느 절에 많은가? 두 논문의 절 순서 차이와 관련이 있어 보이는가?
- 두 논문의 차이를 원인으로 설명할 수 있는가, 아니면 관찰일 뿐인가?

In [17]:
rows = {}
for D, name in ((A, "A"), (B, "B")):
    r = rv.load_run(RUN_ROOT, D.document_id, RUNS[D.document_id])
    for x in rv.aggregate(D, r["whole"], exclude_abstract=True)["rows"]:
        rows.setdefault(ROLE_KO[x["role"]], {})[f"{name}·{KO[x['section_type']]}"] = x["per_1000_words"]
try:
    import pandas as pd
    display(pd.DataFrame(rows).T.fillna("–"))
except ImportError:
    for k, v in rows.items(): print(k, v)

,A·서론,A·방법,A·결과·논의,A·결론,B·서론,B·방법,B·결과·논의,B·결론
선행연구 보고,8.80,6.17,5.08,0.00,3.48,5.44,3.25,0.00
자기 결과 제시,0.00,0.77,9.84,6.94,2.61,0.00,6.50,14.29
해석·주장,1.76,0.77,2.86,6.94,0.87,0.00,4.33,0.00
연구 수행·방법,7.04,36.24,12.06,27.78,12.17,38.10,9.75,4.76
그림·표·본문 지시,0.00,0.77,2.22,0.00,0.87,0.00,1.63,0.00
역할 검토 필요,1.76,2.31,0.95,0.00,1.74,1.48,3.79,0.00


---
# 검토 메모 남기기
실습 중 판정이 의심스러웠던 문장을 기록합니다. 이 메모는 **정답표가 아니라** 다음 회의와 버셀 세션을 위한 검토 기록입니다. 기록할 때마다 셀을 다시 실행하면 한 줄씩 쌓입니다. 칸을 모두 비워 두면 기록 없이 넘어가므로 "모두 실행"을 해도 멈추지 않습니다. 일부만 채우면 기록하지 않고 알려 줍니다.

In [18]:
import csv, os, time
MEMO = "검토메모.csv"
# ── 기록할 내용(직접 채워 주세요) ───────────────
memo_paper    = "paper_a"   # paper_a 또는 paper_b
memo_id       = "3.3.2-020"          # 문장 번호, 예: "3.3.2-020"
memo_reviewer = "시험"          # 황 / 윤
memo_opinion  = "시험 기록"          # 예: "규칙의 판정과 같음", "해석·주장이 아니라 자기 결과로 보임"
memo_reason   = ""          # 화학적·언어학적 근거

if not (memo_id or memo_reviewer or memo_opinion or memo_reason):
    print("기록할 내용이 비어 있어 건너뜁니다. 위 칸을 채운 뒤 이 셀을 다시 실행하면 한 줄씩 기록됩니다.")
elif not (memo_id and memo_reviewer and memo_opinion):
    raise ValueError("memo_id, memo_reviewer, memo_opinion을 모두 채워 주세요. 기록하지 않았습니다. 칸을 고친 뒤 이 셀부터 다시 실행하고, 마지막 내려받기 셀까지 진행하세요.")
else:
  D_ = A if memo_paper == "paper_a" else B
  if memo_id not in D_.by_id(): raise ValueError(f"{memo_paper}에 없는 문장 번호입니다: {memo_id}")
  r_ = rv.load_run(RUN_ROOT, memo_paper, RUNS[memo_paper])
  rule = "; ".join(f'{h["text"]}={ROLE_KO[h["role"]]}' for h in r_["whole"] if h["sentence_id"] == memo_id) or "(색칠한 표현 없음)"
  new = not os.path.exists(MEMO)
  with open(MEMO, "a", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    if new: w.writerow(["시각", "논문", "문서 판본", "실행 ID", "문장번호", "검토자", "규칙의 판정", "검토 의견", "근거", "문장"])
    w.writerow([time.strftime("%Y-%m-%d %H:%M"), memo_paper, D_.document_version, RUNS[memo_paper], memo_id, memo_reviewer,
                rule, memo_opinion, memo_reason, D_.by_id()[memo_id].text])
  print("기록했습니다:", memo_id, "|", rule)
  print("지금까지", sum(1 for _ in open(MEMO, encoding="utf-8-sig")) - 1, "건 (정답표가 아니라 검토 기록입니다)")

기록했습니다: 3.3.2-020 | indicating=해석·주장
지금까지 1 건 (정답표가 아니라 검토 기록입니다)


---
# 결과와 메모 내려받기
코랩 실행 환경의 파일은 영구 보관되지 않습니다. 분석 결과(`rv_runs`)와 검토 메모를 함께 내려받습니다.

In [19]:
z = shutil.make_archive(f"사전실습결과_{time.strftime('%H%M%S')}", "zip", RUN_ROOT)
print("분석 결과:", z)
try:
    from google.colab import files
    files.download(z)
    if os.path.exists("검토메모.csv"): files.download("검토메모.csv")
except ImportError:
    pass

분석 결과: /private/var/folders/6w/dh825y0s1l59l9sysp_1n4h00000gn/T/nbtest_v9skd0vz/사전실습결과_202747.zip


---
# 실습 결과와 후속 확인 사항

| 실습 | 확인한 것 | 후속 확인 |
|---|---|---|
| 1 | 한 문장의 판정 방식 | 판정이 흔들린 예문 기록 |
| 2 | 같은 동사, 다른 역할 | 세 목소리 외의 범주가 필요한지 |
| 3 | 인용 번호 정규화 | 저널별 표기 차이의 영향 |
| 4 | 논문 전체 역할 분포 | 화학 직관과 다른 분포의 원인 |
| 5 | 콘코던스 | 규칙의 일관성, 필요한 문맥 범위 |
| 6 | 읽기 화면 | 실제 읽기에 도움이 되는지, 버셀 요청 사항 |
| 7 | 두 논문 비교 | 관찰과 해석의 구분 |

**판정 원리**

> 동사만 보지 않고, **누가(주어)** 와 **어디서 왔는지(인용)** 를 함께 본다.

- 다른 연구자 + 인용 번호 → 선행연구 보고
- 이 논문의 자료, *we* → 자기 결과
- Figure·Table → 그림·표 지시

**버셀 세션 전에 정리할 것**
1. 검토 메모에서 판정 규칙을 고쳐야 할 사례
2. 화면에서 꼭 필요한 기능과 없어도 되는 기능
3. 버셀 담당자에게 물을 것: 파일 업로드 방식, 결과 저장 위치, API 키 관리, 여러 연구자의 검토 기록 저장

규칙의 색은 후보입니다. 판정의 타당성은 원문을 함께 읽고 검토하는 두 분의 판단으로 확인됩니다.